In [1]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\navon\OneDrive\Desktop\CrossRank")

meta = pd.read_csv(ROOT / "data/processed/uk_etf_metadata.csv")
prices = pd.read_csv(ROOT / "data/processed/uk_etf_prices.csv", parse_dates=["date"])

print(f"Metadata: {len(meta)} tickers")
print(f"Prices:   {len(prices):,} rows, {prices['ticker'].nunique()} unique tickers")
print(f"Date range: {prices['date'].min().date()} to {prices['date'].max().date()}")

# sanity check — every ticker in prices should have a metadata row, and vice versa
missing_meta = set(prices["ticker"]) - set(meta["ticker"])
missing_prices = set(meta["ticker"]) - set(prices["ticker"])
print(f"\nTickers in prices but not metadata: {len(missing_meta)}")
print(f"Tickers in metadata but not prices: {len(missing_prices)}")

meta.head()

Metadata: 1534 tickers
Prices:   2,707,129 rows, 1534 unique tickers
Date range: 2008-01-02 to 2026-10-05

Tickers in prices but not metadata: 0
Tickers in metadata but not prices: 0


,ticker,isin,name,inception_date,age_in_years,base_currency,quote_currency,price_scale,ter,size
0,CSP1,IE00B5BMR087,iShares Core S&P 500 UCITS ETF USD (Acc),2010-05-19,16.394521,USD,GBp,100.0,0.07,118149.0
1,SWDA,IE00B4L5Y983,iShares Core MSCI World UCITS ETF USD (Acc),2009-09-25,17.041096,USD,GBp,100.0,0.20,112120.0
2,VWRP,IE00BK5BQT80,Vanguard FTSE All-World UCITS ETF (USD) Accumu...,2019-07-23,7.210959,USD,GBP,1.0,0.14,46025.0
3,VUSA,IE00B3XXRP09,Vanguard S&P 500 UCITS ETF (USD) Distributing,2012-05-22,14.383562,USD,GBP,1.0,0.07,40335.0
4,EMIM,IE00BKM4GZ66,iShares Core MSCI Emerging Markets IMI UCITS E...,2014-05-30,12.361644,USD,GBp,100.0,0.18,34259.0


In [2]:
import yfinance as yf

# URTH = iShares MSCI World ETF (global equity benchmark, US-listed, independent
# of our LSE universe — avoids circularity of using one of our own funds as benchmark)
benchmark = yf.download("URTH", period="max", auto_adjust=True, progress=False)
if isinstance(benchmark.columns, pd.MultiIndex):
    benchmark.columns = benchmark.columns.get_level_values(0)

benchmark = benchmark[["Close"]].rename(columns={"Close": "benchmark_close"})
benchmark.index.name = "date"
benchmark = benchmark.reset_index()

print(f"Benchmark rows: {len(benchmark)}")
print(f"Date range: {benchmark['date'].min().date()} to {benchmark['date'].max().date()}")
benchmark.tail()

Benchmark rows: 3706
Date range: 2012-01-12 to 2026-10-08


Price,date,benchmark_close
3701,2026-10-02,207.529999
3702,2026-10-05,208.550003
3703,2026-10-06,209.729996
3704,2026-10-07,208.669998
3705,2026-10-08,207.630005


In [3]:
benchmark = yf.download("ACWI", period="max", auto_adjust=True, progress=False)
if isinstance(benchmark.columns, pd.MultiIndex):
    benchmark.columns = benchmark.columns.get_level_values(0)

benchmark = benchmark[["Close"]].rename(columns={"Close": "benchmark_close"})
benchmark.index.name = "date"
benchmark = benchmark.reset_index()
benchmark.columns.name = None  # clear the leftover "Price" column-index label

print(f"Benchmark rows: {len(benchmark)}")
print(f"Date range: {benchmark['date'].min().date()} to {benchmark['date'].max().date()}")
print(f"Our price data starts: {prices['date'].min().date()}")
benchmark.head()

Benchmark rows: 4663
Date range: 2008-03-28 to 2026-10-08
Our price data starts: 2008-01-02


,date,benchmark_close
0,2008-03-28,35.438728
1,2008-03-31,34.886997
2,2008-04-01,35.827778
3,2008-04-02,36.223892
4,2008-04-03,36.429035


In [4]:
# monthly close per ticker (last trading day of each month)
prices_monthly = (
    prices.set_index("date")
    .groupby("ticker")["close"]
    .resample("ME")
    .last()
    .reset_index()
)

prices_monthly["ret"] = prices_monthly.groupby("ticker")["close"].pct_change()

# benchmark monthly return
bench_monthly = (
    benchmark.set_index("date")["benchmark_close"]
    .resample("ME")
    .last()
    .pct_change()
    .reset_index()
    .rename(columns={"benchmark_close": "benchmark_ret"})
)

panel = prices_monthly.merge(bench_monthly, on="date", how="inner")
panel = panel.dropna(subset=["ret", "benchmark_ret"])

print(f"Panel rows: {len(panel):,}")
print(f"Unique tickers: {panel['ticker'].nunique()}")
print(f"Date range: {panel['date'].min().date()} to {panel['date'].max().date()}")
panel.head(10)

Panel rows: 129,282
Unique tickers: 1523
Date range: 2008-04-30 to 2026-10-31


,ticker,date,close,ret,benchmark_ret
1,100D,2017-12-31,115.628760,0.034353,0.014503
2,100D,2018-01-31,113.352275,-0.019688,0.057012
3,100D,2018-02-28,109.493936,-0.034038,-0.044750
4,100D,2018-03-31,107.262949,-0.020375,-0.014975
5,100D,2018-04-30,114.590830,0.068317,0.004045
6,100D,2018-05-31,117.784502,0.027870,0.004723
7,100D,2018-06-30,117.502207,-0.002397,-0.005853
8,100D,2018-07-31,116.556514,-0.008048,0.030652
9,100D,2018-08-31,112.703447,-0.033057,0.006958
10,100D,2018-09-30,114.027197,0.011745,0.006097


In [5]:
panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)

def rolling_beta_r2(g):
    cov = g["ret"].rolling(36, min_periods=24).cov(g["benchmark_ret"])
    var = g["benchmark_ret"].rolling(36, min_periods=24).var()
    corr = g["ret"].rolling(36, min_periods=24).corr(g["benchmark_ret"])
    g = g.copy()
    g["beta"] = cov / var
    g["r2"] = corr ** 2
    return g

panel = panel.groupby("ticker", group_keys=False).apply(rolling_beta_r2)

print(f"Rows with computed beta/r2: {panel['beta'].notna().sum():,} of {len(panel):,}")
panel.dropna(subset=["beta","r2"]).sample(10, random_state=42)

Rows with computed beta/r2: 99,085 of 129,282


,date,close,ret,benchmark_ret,beta,r2
77223,2023-10-31,82.470001,-0.076380,-0.025441,1.019737,0.637044
52997,2020-10-31,4.331500,-0.042974,-0.022272,0.656774,0.572931
67416,2024-08-31,118.850000,0.027226,0.024972,0.431266,0.577458
57725,2014-03-31,21.270913,0.004714,0.005708,0.604460,0.619158
119483,2016-09-30,46.799999,0.005803,0.009393,1.003229,0.955492
45581,2013-07-31,14.018588,0.081289,0.045800,1.004360,0.730417
65627,2023-12-31,5.597500,0.036766,0.048128,0.527214,0.861808
1547,2023-07-31,52.415000,0.043812,0.035960,0.423466,0.271072
34985,2021-02-28,31.692666,0.041462,0.022888,0.777301,0.726142
110091,2016-02-29,10.384254,-0.009524,-0.012485,0.578020,0.214237


In [6]:
R2_THRESHOLD = 0.50
BETA_THRESHOLD = 0.40

panel["equity_like"] = (panel["r2"] >= R2_THRESHOLD) & (panel["beta"] >= BETA_THRESHOLD)

# the key diagnostic: how big is our eligible universe, month by month, over time?
universe_size = panel.groupby("date")["equity_like"].sum()

print(f"Total (ticker, month) observations classified: {panel['beta'].notna().sum():,}")
print(f"Equity-like observations: {panel['equity_like'].sum():,} "
      f"({panel['equity_like'].sum() / panel['beta'].notna().sum() * 100:.1f}%)")

print("\nEligible universe size by year (Dec snapshot):")
yearly_snapshot = universe_size[universe_size.index.month == 12]
print(yearly_snapshot)

Total (ticker, month) observations classified: 99,085
Equity-like observations: 40,546 (40.9%)

Eligible universe size by year (Dec snapshot):
date
2008-12-31      0
2009-12-31      0
2010-12-31      4
2011-12-31     72
2012-12-31     92
2013-12-31     94
2014-12-31     99
2015-12-31     90
2016-12-31     67
2017-12-31     56
2018-12-31     56
2019-12-31    155
2020-12-31    448
2021-12-31    466
2022-12-31    461
2023-12-31    360
2024-12-31    394
2025-12-31    259
Name: equity_like, dtype: int64


In [8]:
# rebuild monthly panel cleanly (previous panel lost its 'ticker' column)
prices_monthly = (
    prices.set_index("date")
    .groupby("ticker")["close"]
    .resample("ME")
    .last()
    .reset_index()
)
prices_monthly["ret"] = prices_monthly.groupby("ticker")["close"].pct_change()

bench_monthly = (
    benchmark.set_index("date")["benchmark_close"]
    .resample("ME")
    .last()
    .pct_change()
    .reset_index()
    .rename(columns={"benchmark_close": "benchmark_ret"})
)

panel = prices_monthly.merge(bench_monthly, on="date", how="inner")
panel = panel.dropna(subset=["ret", "benchmark_ret"]).sort_values(["ticker","date"]).reset_index(drop=True)

# compute beta/r2 as a separate frame, joined back by index — avoids the
# groupby().apply() grouping-column-exclusion issue in pandas 3.0
def compute_beta_r2(g):
    cov = g["ret"].rolling(36, min_periods=24).cov(g["benchmark_ret"])
    var = g["benchmark_ret"].rolling(36, min_periods=24).var()
    corr = g["ret"].rolling(36, min_periods=24).corr(g["benchmark_ret"])
    return pd.DataFrame({"beta": cov / var, "r2": corr ** 2}, index=g.index)

roll = panel.groupby("ticker", group_keys=False).apply(compute_beta_r2)
panel = panel.join(roll)

R2_THRESHOLD = 0.50
BETA_THRESHOLD = 0.40
panel["equity_like"] = (panel["r2"] >= R2_THRESHOLD) & (panel["beta"] >= BETA_THRESHOLD)

print(f"Panel rows: {len(panel):,}  |  tickers: {panel['ticker'].nunique()}")
print(f"Rows with computed beta/r2: {panel['beta'].notna().sum():,}")
print(f"Equity-like observations: {panel['equity_like'].sum():,}")

# the diagnostic we actually wanted
total_active = panel.groupby("date")["ticker"].nunique()
eligible = panel[panel["equity_like"]].groupby("date")["ticker"].nunique()
compare = pd.DataFrame({"total_active": total_active, "equity_like": eligible}).fillna(0)
compare["pass_rate"] = compare["equity_like"] / compare["total_active"]

print(compare[compare.index.month == 12])

Panel rows: 129,282  |  tickers: 1523
Rows with computed beta/r2: 99,085
Equity-like observations: 40,546
            total_active  equity_like  pass_rate
date                                            
2008-12-31             3          0.0   0.000000
2009-12-31           136          0.0   0.000000
2010-12-31           193          4.0   0.020725
2011-12-31           238         72.0   0.302521
2012-12-31           278         92.0   0.330935
2013-12-31           299         94.0   0.314381
2014-12-31           354         99.0   0.279661
2015-12-31           432         90.0   0.208333
2016-12-31           469         67.0   0.142857
2017-12-31           518         56.0   0.108108
2018-12-31           608         56.0   0.092105
2019-12-31           676        155.0   0.229290
2020-12-31           734        448.0   0.610354
2021-12-31           830        466.0   0.561446
2022-12-31           942        461.0   0.489384
2023-12-31          1031        360.0   0.349176
2024-12-31  

In [9]:
import pandas as pd

R2_LEVELS = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
BETA_THRESHOLD = 0.40

results = {}
for r2_thresh in R2_LEVELS:
    eq_like = (panel["r2"] >= r2_thresh) & (panel["beta"] >= BETA_THRESHOLD)
    eligible = panel[eq_like].groupby("date")["ticker"].nunique()
    results[r2_thresh] = eligible

comparison = pd.DataFrame(results)
comparison.columns = [f"r2>={x}" for x in R2_LEVELS]

dec_comparison = comparison[comparison.index.month == 12]
print(dec_comparison)

# stability metric: coefficient of variation of year-over-year pass rate
# (lower = more stable across regimes, which is what we actually want)
pass_rates = dec_comparison.div(total_active[total_active.index.month==12], axis=0)
print("\nCoefficient of variation (lower = more stable):")
print((pass_rates.std() / pass_rates.mean()).round(3))

            r2>=0.15  r2>=0.2  r2>=0.25  r2>=0.3  r2>=0.35  r2>=0.4  r2>=0.5
date                                                                        
2010-12-31         4        4         4        4         4        4        4
2011-12-31       105      100        95       84        82       80       72
2012-12-31       142      129       117      111       106      102       92
2013-12-31       168      153       144      133       125      115       94
2014-12-31       217      198       175      163       155      135       99
2015-12-31       241      219       197      181       151      138       90
2016-12-31       277      257       222      174       150      108       67
2017-12-31       301      245       214      160       127      107       56
2018-12-31       365      311       234      174       139       92       56
2019-12-31       437      419       382      354       297      249      155
2020-12-31       571      568       564      548       521      508      448

In [10]:
R2_FINAL = 0.25
BETA_FINAL = 0.40

panel["equity_like"] = (panel["r2"] >= R2_FINAL) & (panel["beta"] >= BETA_FINAL)

marginal = panel[(panel["equity_like"]) & (panel["r2"] < 0.35) & (panel["date"] == panel["date"].max())]
marginal_named = marginal.merge(meta[["ticker","name"]], on="ticker", how="left")
print(f"Marginal equity-like funds (r2 0.25-0.35) as of latest month: {len(marginal_named)}")
marginal_named[["ticker","name","beta","r2"]].sort_values("r2")

Marginal equity-like funds (r2 0.25-0.35) as of latest month: 121


,ticker,name,beta,r2
5,CS1,Amundi IBEX 35 UCITS ETF Acc,0.562449,0.250227
93,URJP,HANetf Sprott Junior Uranium Miners UCITS ETF ...,2.444315,0.252512
53,JARI,Amundi MSCI Japan SRI Climate Paris Aligned UC...,0.467792,0.252637
30,GNOG,Global X Genomics & Biotechnology UCITS ETF,1.423903,0.253948
84,SMID,First Trust SMID Rising Dividend Achievers UCI...,0.832328,0.254063
...,...,...,...,...
113,XM1D,Xtrackers MSCI Japan UCITS ETF 1D,0.642190,0.346076
85,SMTV,VanEck Morningstar US SMID Moat UCITS ETF A,0.842709,0.346794
111,XLIP,Invesco US Industrials Sector UCITS ETF,0.837848,0.347392
68,LGJG,L&G Japan Equity UCITS ETF,0.631623,0.348800


In [11]:
R2_THRESHOLD = 0.25
BETA_THRESHOLD = 0.40

panel["equity_like"] = (panel["r2"] >= R2_THRESHOLD) & (panel["beta"] >= BETA_THRESHOLD)

print(f"Final equity-like observations: {panel['equity_like'].sum():,} "
      f"of {panel['beta'].notna().sum():,} classified ({panel['equity_like'].sum()/panel['beta'].notna().sum()*100:.1f}%)")

eligible_by_month = panel[panel["equity_like"]].groupby("date")["ticker"].nunique()
print("\nEligible universe, Dec snapshots:")
print(eligible_by_month[eligible_by_month.index.month == 12])

Final equity-like observations: 70,885 of 99,085 classified (71.5%)

Eligible universe, Dec snapshots:
date
2010-12-31      4
2011-12-31     95
2012-12-31    117
2013-12-31    144
2014-12-31    175
2015-12-31    197
2016-12-31    222
2017-12-31    214
2018-12-31    234
2019-12-31    382
2020-12-31    564
2021-12-31    582
2022-12-31    624
2023-12-31    600
2024-12-31    698
2025-12-31    700
Name: ticker, dtype: int64


In [12]:
from pathlib import Path

ROOT = Path(r"C:\Users\navon\OneDrive\Desktop\CrossRank")
out_path = ROOT / "data/processed/uk_etf_monthly_panel.csv"

panel_to_save = panel[["ticker","date","close","ret","benchmark_ret","beta","r2","equity_like"]]
panel_to_save.to_csv(out_path, index=False)

import os
print(f"Saved: {out_path}")
print(f"Size: {round(os.path.getsize(out_path)/1e6, 1)} MB")
print(f"Rows: {len(panel_to_save):,}")

Saved: C:\Users\navon\OneDrive\Desktop\CrossRank\data\processed\uk_etf_monthly_panel.csv
Size: 14.0 MB
Rows: 129,282


In [13]:
# compute trailing 60-day median traded value (price × volume) per ticker, per day
prices_sorted = prices.sort_values(["ticker", "date"]).copy()
prices_sorted["traded_value"] = prices_sorted["close"] * prices_sorted["volume"]
prices_sorted["median_traded_val_60d"] = (
    prices_sorted.groupby("ticker")["traded_value"]
    .transform(lambda s: s.rolling(60, min_periods=30).median())
)
prices_sorted["zero_vol_pct_60d"] = (
    prices_sorted.groupby("ticker")["volume"]
    .transform(lambda s: (s == 0).rolling(60, min_periods=30).mean())
)

# take month-end snapshot to align with the monthly panel
liquidity_monthly = (
    prices_sorted.set_index("date")
    .groupby("ticker")[["median_traded_val_60d", "zero_vol_pct_60d"]]
    .resample("ME")
    .last()
    .reset_index()
)

panel = panel.merge(liquidity_monthly, on=["ticker", "date"], how="left")

print(f"Panel rows after liquidity merge: {len(panel):,}")
print(panel[["median_traded_val_60d","zero_vol_pct_60d"]].describe())

Panel rows after liquidity merge: 129,282
       median_traded_val_60d  zero_vol_pct_60d
count           1.287550e+05     128755.000000
mean            3.155185e+05          0.331994
std             1.817958e+06          0.371643
min             0.000000e+00          0.000000
25%             0.000000e+00          0.000000
50%             1.187332e+04          0.150000
75%             1.126342e+05          0.666667
max             7.214182e+07          1.000000


In [14]:
LIQ_THRESHOLDS = [0, 5_000, 10_000, 25_000, 50_000, 100_000]
ZEROVOL_MAX = 0.30  # exclude anything with >30% zero-volume days in trailing 60d

results = {}
for liq in LIQ_THRESHOLDS:
    mask = (
        panel["equity_like"]
        & (panel["median_traded_val_60d"] >= liq)
        & (panel["zero_vol_pct_60d"] <= ZEROVOL_MAX)
    )
    results[liq] = panel[mask].groupby("date")["ticker"].nunique()

liq_comparison = pd.DataFrame(results)
liq_comparison.columns = [f"liq>={x}" for x in LIQ_THRESHOLDS]

dec_liq = liq_comparison[liq_comparison.index.month == 12]
print(dec_liq)

            liq>=0  liq>=5000  liq>=10000  liq>=25000  liq>=50000  liq>=100000
date                                                                          
2010-12-31       1          1           1           1           1            1
2011-12-31      49         49          49          42          37           33
2012-12-31      54         52          50          41          34           25
2013-12-31      63         62          60          54          48           37
2014-12-31      83         81          78          66          56           47
2015-12-31      97         93          88          78          63           47
2016-12-31     104        101          95          84          79           57
2017-12-31     102         97          94          83          69           56
2018-12-31     111        109         101          80          69           54
2019-12-31     196        188         175         146         119           91
2020-12-31     360        347         325         27

In [15]:
LIQ_FINAL = 10_000
ZEROVOL_FINAL = 0.30

panel["eligible"] = (
    panel["equity_like"]
    & (panel["median_traded_val_60d"] >= LIQ_FINAL)
    & (panel["zero_vol_pct_60d"] <= ZEROVOL_FINAL)
)

final_universe = panel[panel["eligible"]].groupby("date")["ticker"].nunique()
print("Final eligible universe size, Dec snapshots:")
print(final_universe[final_universe.index.month == 12])
print(f"\nTotal eligible (ticker, month) observations: {panel['eligible'].sum():,}")

Final eligible universe size, Dec snapshots:
date
2010-12-31      1
2011-12-31     49
2012-12-31     50
2013-12-31     60
2014-12-31     78
2015-12-31     88
2016-12-31     95
2017-12-31     94
2018-12-31    101
2019-12-31    175
2020-12-31    325
2021-12-31    330
2022-12-31    322
2023-12-31    326
2024-12-31    433
2025-12-31    475
Name: ticker, dtype: int64

Total eligible (ticker, month) observations: 40,144


In [16]:
out_path = ROOT / "data/processed/uk_etf_eligible_panel.csv"
panel.to_csv(out_path, index=False)

import os
print(f"Saved: {out_path}")
print(f"Size: {round(os.path.getsize(out_path)/1e6, 1)} MB")
print(f"Rows: {len(panel):,}")
print(f"Columns: {list(panel.columns)}")

Saved: C:\Users\navon\OneDrive\Desktop\CrossRank\data\processed\uk_etf_eligible_panel.csv
Size: 17.7 MB
Rows: 129,282
Columns: ['ticker', 'date', 'close', 'ret', 'benchmark_ret', 'beta', 'r2', 'equity_like', 'median_traded_val_60d', 'zero_vol_pct_60d', 'eligible']
